In [1]:
import pandas as pd
loans_cleaned_raw = pd.read_csv('../data/library_loans_clean.csv', parse_dates=['loan_date','return_date'])
loans_merged_raw = pd.read_csv('../data/library_loans_clean_merged.csv', parse_dates=['loan_date','return_date'])
loans_raw = loans_cleaned_raw.copy()
loans = loans_merged_raw.copy()
members_raw = pd.read_csv('../data/members_clean.csv')
members = members_raw.copy()

# 1)

In [2]:
loans._merge.value_counts()

_merge
both         569
left_only     31
Name: count, dtype: int64

In [3]:
print(len(loans[loans.branch == 'no member record'])/len(loans))

0.051666666666666666


In [4]:
print(
    loans.where(loans.branch == 'no member record')['late_fee'].sum(),
    loans.where(loans.branch == 'no member record')['late_fee'].sum() /
    loans.late_fee.sum()
)

347.5 0.04641068447412354


In [5]:
loans.where(loans.branch == 'no member record').groupby('genre')['loan_id'].count().sort_values(ascending=False)

genre
Children    10
Sci-fi       5
Crime        4
Fantasy      4
History      4
Romance      4
Name: loan_id, dtype: int64

In [6]:
loans.groupby('genre')['loan_id'].count().sort_values(ascending=False)

genre
Crime       155
Children    130
Sci-fi       89
Fantasy      87
Romance      82
History      57
Name: loan_id, dtype: int64

Children genre is overrepresented and Crime underrepresented, but overall numbers are small, so there's a great uncertainty in the numbers.

Would ask whoever made the members table if they know why there are missing member_ids... have they been deleted or is the members table not up to date?

# 2)

In [7]:
outer = loans_raw.merge(members, how='outer', validate='many_to_one', indicator=True)

In [8]:
outer.shape

(601, 11)

In [9]:
outer._merge.value_counts()

_merge
both          569
left_only      31
right_only      1
Name: count, dtype: int64

In [13]:
print(len(members[members.member_id.isin(loans.member_id) == False]))

1


In [14]:
members[members.member_id.isin(loans.member_id) == False]

,member_id,branch,birth_year
47,M0050,Harbour,1999.0


Would prefer the isin way, that way we are not creating an extra dataframe just to check something.

# 3)

In [22]:
age_at_loan = loans.loan_date.dt.year - loans.birth_year
print(len(age_at_loan[age_at_loan.isna()]))

143


In [34]:
print(len(loans[loans.loan_date.dt.year.isna()]))
print(len(loans[loans.birth_year.isna()]))

3
140


In [39]:
age_at_loan.groupby(loans.genre).mean().sort_values().round(1)

genre
History     35.2
Sci-fi      37.8
Romance     38.1
Crime       38.4
Fantasy     39.8
Children    41.1
dtype: float64

Children books have the highest average age (we'll assume most people who loan children books are parents).
All in all the ages don't wary a lot, there's not much to conclude.
History books have the lowest average age.

# 4)

Problems:
* 18 missing birth_years (left alone)
* 8 birth_years are 1900 (removed birth_year)
* 3 birth_years are 2030 (removed birth_year)
* 4 members are duplicates (removed duplicates)
* members table is missing 'members_id's that exist in the loans table (left alone)

# 5)

Which birth year has the most number of loans total?

In [42]:
loans.groupby('birth_year')['loan_id'].count().sort_values(ascending=False)

birth_year
1990.0    35
1979.0    35
1987.0    20
1992.0    19
1976.0    16
1996.0    16
1986.0    14
1984.0    14
1985.0    14
1975.0    13
1999.0    13
2004.0    13
1994.0    13
1968.0    12
1988.0    12
1981.0    12
1971.0    11
1991.0    11
1978.0    11
1974.0    10
1997.0    10
2010.0    10
2005.0    10
1982.0     9
1967.0     9
2001.0     9
1998.0     8
2003.0     7
2009.0     6
1993.0     6
2007.0     6
1989.0     6
1980.0     6
1995.0     5
2000.0     5
1983.0     5
2002.0     5
1977.0     4
1957.0     4
1970.0     4
1962.0     4
1961.0     4
1960.0     2
2008.0     2
Name: loan_id, dtype: int64